# INC-008 Chronotype Cohort Review

This notebook inspects an existing pipeline run against the INC-008 contract (`specs/INC-008.md`): the `gold_chronotype_cohort` participant table, its cohort attribute, coverage columns, the 14-valid-day availability threshold, its consistency with `gold_clinical_daily` and silver, and its idempotent and incremental behavior. The supplied database is opened read-only. The rerun and incremental sections work on temporary copies. Raw inputs are read only.

Generate the run first with `python -m wearable_pipeline run --data-dir data --output-dir output/runs/inc008`. Set `INC008_RUN_DIR` if the output directory differs, `INC008_DATA_DIR` to override `data/`, and `INC008_PREVIOUS_RUN_DIR` (default `output/runs/inc007`, a run before the cohort table) to confirm the existing daily and weekly tables are unchanged. Sections 8 and 9 run two full rebuilds and one incremental run (about 4 minutes).

## Review Sections
1. Environment, paths and display helpers.
2. Row counts, schema, key and cohort attribute.
3. Coverage columns.
4. Cohort values reconciled with `gold_clinical_daily`.
5. Steps and heart-rate means recomputed independently from silver.
6. The availability threshold (`MIN_VALID_DAYS = 14`) and its use on this dataset.
7. Daily and weekly gold versus a previous run (optional).
8. Rerun on an isolated copy: identical tables and report; raw data unchanged.
9. One corrected record: incremental versus a full rebuild changes exactly one cohort row.
10. Acceptance summary with PASS/FAIL/NOT RUN.

Aggregates and a few participant keys are displayed; raw record values are not printed.

In [ ]:
import hashlib
import html
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path

import duckdb
from IPython.display import HTML, display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'src' / 'wearable_pipeline').is_dir()
)
sys.path.insert(0, str(ROOT / 'src'))
from wearable_pipeline.cohort import COHORT_COLUMNS, MIN_VALID_DAYS
from wearable_pipeline.compare import compare_outputs, format_comparison
from wearable_pipeline.incremental import run_incremental
from wearable_pipeline.pipeline import run_pipeline

DATA_DIR = Path(os.environ.get('INC008_DATA_DIR', ROOT / 'data')).resolve()
RUN_DIR = Path(os.environ.get('INC008_RUN_DIR', ROOT / 'output' / 'runs' / 'inc008')).resolve()
PREVIOUS_RUN_DIR = Path(
    os.environ.get('INC008_PREVIOUS_RUN_DIR', ROOT / 'output' / 'runs' / 'inc007')
).resolve()
DATABASE_PATH = RUN_DIR / 'wearable.duckdb'
REPORT_PATH = RUN_DIR / 'quality' / 'report.md'

if not DATABASE_PATH.is_file():
    raise FileNotFoundError(f'DuckDB file not found: {DATABASE_PATH}. Set INC008_RUN_DIR.')
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Raw data directory not found: {DATA_DIR}')

connection = duckdb.connect(str(DATABASE_PATH), read_only=True)
acceptance = {}


def render_table(title, columns, rows):
    header = ''.join('<th>' + html.escape(str(column)) + '</th>' for column in columns)
    body = ''.join(
        '<tr>' + ''.join('<td>' + html.escape(str(value)) + '</td>' for value in row) + '</tr>'
        for row in rows
    )
    display(HTML(
        '<h4>' + html.escape(title) + '</h4><table><thead><tr>' + header
        + '</tr></thead><tbody>' + body + '</tbody></table>'
    ))


def show_query(title, sql, parameters=None, db=connection):
    cursor = db.execute(sql, parameters or [])
    render_table(title, [item[0] for item in cursor.description], cursor.fetchall())


def status(ok):
    return 'PASS' if ok else 'FAIL'


def tree_hash(path):
    digest = hashlib.sha256()
    for file in sorted(Path(path).rglob('*')):
        if file.is_file():
            digest.update(str(file.relative_to(path)).encode())
            digest.update(hashlib.sha256(file.read_bytes()).digest())
    return digest.hexdigest()


report_text = REPORT_PATH.read_text(encoding='utf-8')
run_id = next(
    (line.split(': ', 1)[1] for line in report_text.splitlines() if line.startswith('Run id: ')),
    None,
)
RAW_HASH_BEFORE = tree_hash(DATA_DIR)
print(f'Database: {DATABASE_PATH}')
print(f'Data directory: {DATA_DIR}')
print(f'Run ID: {run_id}')
print(f'Raw data fingerprint before: {RAW_HASH_BEFORE[:16]}')
print(f'DuckDB version: {duckdb.__version__}')

In [ ]:
# Section 2: row counts, schema, key and cohort attribute.
cohort_rows = connection.execute('SELECT COUNT(*) FROM gold_chronotype_cohort').fetchone()[0]
distinct_keys = connection.execute(
    'SELECT COUNT(DISTINCT participant_id) FROM gold_chronotype_cohort').fetchone()[0]
enrolled = connection.execute(
    'SELECT COUNT(DISTINCT participant_id) FROM silver_participants').fetchone()[0]
orphans = connection.execute(
    'SELECT COUNT(*) FROM gold_chronotype_cohort WHERE participant_id NOT IN '
    '(SELECT participant_id FROM silver_participants)').fetchone()[0]
render_table(
    'Row counts',
    ['check', 'expected', 'observed', 'status'],
    [
        ('gold_chronotype_cohort rows (one per enrolled participant)', enrolled, cohort_rows, status(cohort_rows == enrolled == 50)),
        ('distinct participant_id keys', cohort_rows, distinct_keys, status(distinct_keys == cohort_rows)),
        ('rows for participants that are not enrolled', 0, orphans, status(orphans == 0)),
    ],
)
schema = connection.execute(
    '''SELECT column_name, data_type FROM information_schema.columns
       WHERE table_name = 'gold_chronotype_cohort' ORDER BY ordinal_position'''
).fetchall()
expected_schema = [(name, kind) for name, kind in COHORT_COLUMNS]
render_table(
    'Cohort schema (observed)',
    ['position', 'column', 'type'],
    [(i + 1, name, kind) for i, (name, kind) in enumerate(schema)],
)
show_query(
    'Cohort sizes and flags',
    '''SELECT chronotype_cohort, cohort_flag, COUNT(*) AS participants
       FROM gold_chronotype_cohort GROUP BY ALL ORDER BY 1, 2''',
)
sizes = dict(connection.execute(
    'SELECT chronotype_cohort, COUNT(*) FROM gold_chronotype_cohort GROUP BY 1').fetchall())
attribute_mismatch = connection.execute(
    '''SELECT COUNT(*) FROM gold_chronotype_cohort c JOIN silver_participants p USING (participant_id)
       WHERE c.chronotype_cohort IS DISTINCT FROM p.chronotype''').fetchone()[0]
bad_flag = connection.execute(
    '''SELECT COUNT(*) FROM gold_chronotype_cohort
       WHERE (chronotype_cohort IS NULL) <> (cohort_flag IS NOT NULL)''').fetchone()[0]
render_table(
    'Schema and cohort checks',
    ['check', 'observed', 'status'],
    [
        ('Columns and types equal the specified 13', len(schema), status(schema == expected_schema and len(schema) == 13)),
        ('Cohort sizes A / B (profiling: 28 / 22)', (sizes.get('A'), sizes.get('B')), status(sizes.get('A') == 28 and sizes.get('B') == 22)),
        ('Cohort differs from silver_participants.chronotype', attribute_mismatch, status(attribute_mismatch == 0)),
        ('Rows where NULL cohort and flag disagree', bad_flag, status(bad_flag == 0)),
    ],
)
acceptance['50 rows, one per enrolled participant, unique participant_id, no orphan rows'] = (
    cohort_rows == 50 and distinct_keys == 50 and enrolled == 50 and orphans == 0)
acceptance['Cohort schema has the 13 specified columns'] = schema == expected_schema
acceptance['Cohorts A 28 and B 22, equal to silver_participants, flag only when the cohort is NULL'] = (
    sizes.get('A') == 28 and sizes.get('B') == 22 and attribute_mismatch == 0 and bad_flag == 0)

In [ ]:
# Section 3: coverage columns (never NULL) and their relation to the daily rows.
show_query(
    'Coverage distribution (participants per valid-day count)',
    '''SELECT 'sleep_valid_days' AS column_name, sleep_valid_days AS value, COUNT(*) AS participants FROM gold_chronotype_cohort GROUP BY ALL
       UNION ALL SELECT 'sleep_stage_valid_days', sleep_stage_valid_days, COUNT(*) FROM gold_chronotype_cohort GROUP BY ALL
       UNION ALL SELECT 'steps_valid_days', steps_valid_days, COUNT(*) FROM gold_chronotype_cohort GROUP BY ALL
       UNION ALL SELECT 'hr_valid_days', hr_valid_days, COUNT(*) FROM gold_chronotype_cohort GROUP BY ALL
       ORDER BY 1, 2''',
)
show_query(
    'Participants with fewer than window_days steps or heart-rate valid days',
    '''SELECT participant_id, chronotype_cohort, steps_valid_days, hr_valid_days
       FROM gold_chronotype_cohort WHERE steps_valid_days < window_days OR hr_valid_days < window_days
       ORDER BY 1''',
)
window_days_ok = connection.execute(
    'SELECT COUNT(*) FROM gold_chronotype_cohort WHERE window_days IS DISTINCT FROM (SELECT study_days FROM run_window)').fetchone()[0]
null_coverage = connection.execute(
    '''SELECT COUNT(*) FROM gold_chronotype_cohort WHERE window_days IS NULL OR sleep_valid_days IS NULL
       OR sleep_stage_valid_days IS NULL OR steps_valid_days IS NULL OR hr_valid_days IS NULL''').fetchone()[0]
out_of_range = connection.execute(
    '''SELECT COUNT(*) FROM gold_chronotype_cohort WHERE sleep_valid_days NOT BETWEEN 0 AND window_days
       OR sleep_stage_valid_days NOT BETWEEN 0 AND window_days OR steps_valid_days NOT BETWEEN 0 AND window_days
       OR hr_valid_days NOT BETWEEN 0 AND window_days''').fetchone()[0]
render_table(
    'Coverage checks',
    ['check', 'observed', 'status'],
    [
        ('window_days differs from run_window.study_days', window_days_ok, status(window_days_ok == 0)),
        ('Coverage columns that are NULL', null_coverage, status(null_coverage == 0)),
        ('Valid-day counts outside 0 to window_days', out_of_range, status(out_of_range == 0)),
    ],
)
acceptance['window_days equals the run window and valid-day counts are never NULL and within 0 to window_days'] = (
    window_days_ok == 0 and null_coverage == 0 and out_of_range == 0)

In [ ]:
# Section 4: cohort values reconciled with gold_clinical_daily (same days, same rules).
daily = '''(SELECT participant_id,
       COUNT(total_sleep_min) AS sleep_days, COUNT(deep_sleep_pct) AS stage_days, COUNT(total_steps) AS steps_days,
       COUNT(*) FILTER (WHERE hr_valid_minute_count > 0) AS hr_days,
       AVG(total_sleep_min) AS sleep_mean, AVG(sleep_efficiency_pct) AS eff_mean, AVG(deep_sleep_pct) AS deep_mean,
       AVG(total_steps) AS steps_mean, AVG(hr_mean_bpm) AS hr_mean
     FROM gold_clinical_daily GROUP BY 1)'''
reconciliation = {
    'sleep_valid_days = days with a daily total sleep': 'c.sleep_valid_days <> d.sleep_days',
    'sleep_stage_valid_days = days with a daily deep-sleep share': 'c.sleep_stage_valid_days <> d.stage_days',
    'steps_valid_days = days with a daily total steps': 'c.steps_valid_days <> d.steps_days',
    'hr_valid_days = days with heart-rate minutes': 'c.hr_valid_days <> d.hr_days',
    'sleep_total_min_mean = mean of the daily totals': 'c.sleep_total_min_mean IS NOT NULL AND ABS(c.sleep_total_min_mean - d.sleep_mean) > 1e-9',
    'sleep_efficiency_pct_mean = mean of the daily values': 'c.sleep_efficiency_pct_mean IS NOT NULL AND ABS(c.sleep_efficiency_pct_mean - d.eff_mean) > 1e-9',
    'deep_sleep_pct_mean = mean of the daily values': 'c.deep_sleep_pct_mean IS NOT NULL AND ABS(c.deep_sleep_pct_mean - d.deep_mean) > 1e-9',
    'steps_daily_mean = mean of the daily totals': 'c.steps_daily_mean IS NOT NULL AND ABS(c.steps_daily_mean - d.steps_mean) > 1e-9',
    'hr_daily_mean_bpm = mean of the daily means (each day weighs equally)': 'c.hr_daily_mean_bpm IS NOT NULL AND ABS(c.hr_daily_mean_bpm - d.hr_mean) > 1e-9',
}
recon_rows = []
for name, condition in reconciliation.items():
    mismatches = connection.execute(
        'SELECT COUNT(*) FROM gold_chronotype_cohort c JOIN ' + daily + ' d USING (participant_id) WHERE ' + condition
    ).fetchone()[0]
    recon_rows.append((name, mismatches, status(mismatches == 0)))
render_table('Cohort versus daily reconciliation (all participants)', ['check', 'mismatching rows', 'status'], recon_rows)
show_query(
    'Participant heart-rate mean is a mean of daily means, not minute-weighted (largest differences against the weekly table)',
    '''SELECT c.participant_id, ROUND(c.hr_daily_mean_bpm, 6) AS cohort_daily_mean,
              ROUND(w.minute_weighted, 6) AS minute_weighted_from_weekly,
              ROUND(ABS(c.hr_daily_mean_bpm - w.minute_weighted), 6) AS difference
       FROM gold_chronotype_cohort c JOIN (
           SELECT participant_id, SUM(hr_weekly_mean_bpm * hr_valid_minutes) / SUM(hr_valid_minutes) AS minute_weighted
           FROM gold_biostat_weekly WHERE hr_weekly_mean_bpm IS NOT NULL GROUP BY 1) w USING (participant_id)
       ORDER BY difference DESC, 1 LIMIT 5''',
)
show_query(
    'Cohort means (information only; the comparison query itself belongs to INC-010)',
    '''SELECT chronotype_cohort, COUNT(*) AS participants,
              ROUND(AVG(sleep_total_min_mean), 2) AS sleep_total_min, ROUND(AVG(steps_daily_mean), 1) AS steps_daily,
              ROUND(AVG(hr_daily_mean_bpm), 2) AS hr_bpm
       FROM gold_chronotype_cohort GROUP BY 1 ORDER BY 1''',
)
acceptance['Cohort counts and means reconcile with gold_clinical_daily for every participant'] = all(
    row[1] == 0 for row in recon_rows)

In [ ]:
# Section 5: steps and heart-rate means recomputed independently from silver (not from gold).
# One value per minute slot (timestamp truncated to the minute): exact duplicates count once, a conflicting
# slot is already NULL in silver. Orphan, out-of-window, undated and folder-mismatch rows never contribute.
silver_check = connection.execute('''
    WITH steps_slot AS (
        SELECT participant_id, local_date, DATE_TRUNC('minute', minute_ts) AS minute_ts, MIN(steps_per_minute) AS v
        FROM silver_steps_minute
        WHERE NOT is_orphan AND NOT is_out_of_window AND local_date IS NOT NULL
          AND COALESCE(quality_flags, '') NOT LIKE '%FOLDER_MISMATCH%'
        GROUP BY ALL),
    steps_day AS (
        SELECT participant_id, local_date, SUM(v) AS total FROM steps_slot GROUP BY 1, 2 HAVING COUNT(v) > 0),
    steps_p AS (
        SELECT participant_id, COUNT(*) AS days, AVG(total) AS mean FROM steps_day GROUP BY 1),
    hr_slot AS (
        SELECT participant_id, local_date, DATE_TRUNC('minute', minute_ts) AS minute_ts, MIN(heart_rate_bpm_normalized) AS v
        FROM silver_heart_rate_minute
        WHERE NOT is_orphan AND NOT is_out_of_window AND local_date IS NOT NULL
          AND COALESCE(quality_flags, '') NOT LIKE '%FOLDER_MISMATCH%'
        GROUP BY ALL),
    hr_day AS (
        SELECT participant_id, local_date, SUM(v) / COUNT(v) AS day_mean FROM hr_slot GROUP BY 1, 2 HAVING COUNT(v) > 0),
    hr_p AS (
        SELECT participant_id, COUNT(*) AS days, AVG(day_mean) AS mean FROM hr_day GROUP BY 1)
    SELECT
        COUNT(*) FILTER (WHERE c.steps_valid_days IS DISTINCT FROM COALESCE(s.days, 0)),
        COUNT(*) FILTER (WHERE c.steps_daily_mean IS NOT NULL AND ABS(c.steps_daily_mean - s.mean) > 1e-9),
        COUNT(*) FILTER (WHERE c.hr_valid_days IS DISTINCT FROM COALESCE(h.days, 0)),
        COUNT(*) FILTER (WHERE c.hr_daily_mean_bpm IS NOT NULL AND ABS(c.hr_daily_mean_bpm - h.mean) > 1e-9),
        COUNT(*)
    FROM gold_chronotype_cohort c
    LEFT JOIN steps_p s USING (participant_id)
    LEFT JOIN hr_p h USING (participant_id)''').fetchone()
labels = ['steps_valid_days', 'steps_daily_mean', 'hr_valid_days', 'hr_daily_mean_bpm']
render_table(
    'Independent silver recomputation (participants: %d)' % silver_check[4],
    ['cohort column', 'mismatching rows', 'status'],
    [(label, count, status(count == 0)) for label, count in zip(labels, silver_check[:4])],
)
acceptance['Steps and heart-rate means and valid days equal an independent recomputation from silver'] = (
    all(count == 0 for count in silver_check[:4]) and silver_check[4] == 50)

In [ ]:
# Section 6: the availability threshold. A mean is NULL exactly when its own valid-day count is below MIN_VALID_DAYS.
gates = {
    'sleep_total_min_mean': 'sleep_valid_days',
    'sleep_efficiency_pct_mean': 'sleep_stage_valid_days',
    'deep_sleep_pct_mean': 'sleep_stage_valid_days',
    'steps_daily_mean': 'steps_valid_days',
    'hr_daily_mean_bpm': 'hr_valid_days',
}
threshold_rows = []
for measure, gate in gates.items():
    wrong = connection.execute(
        f'SELECT COUNT(*) FROM gold_chronotype_cohort WHERE ({measure} IS NULL) <> ({gate} < {MIN_VALID_DAYS})'
    ).fetchone()[0]
    null_rows = connection.execute(f'SELECT COUNT(*) FROM gold_chronotype_cohort WHERE {measure} IS NULL').fetchone()[0]
    threshold_rows.append((measure, gate, null_rows, wrong, status(wrong == 0)))
render_table(
    f'Availability threshold (MIN_VALID_DAYS = {MIN_VALID_DAYS})',
    ['measure', 'gate column', 'rows with NULL measure', 'rows violating the rule', 'status'],
    threshold_rows,
)
below = connection.execute(
    f'''SELECT COUNT(*) FROM gold_chronotype_cohort WHERE sleep_valid_days < {MIN_VALID_DAYS}
        OR sleep_stage_valid_days < {MIN_VALID_DAYS} OR steps_valid_days < {MIN_VALID_DAYS} OR hr_valid_days < {MIN_VALID_DAYS}''').fetchone()[0]
print(f'Participants below the threshold for any measure on this dataset: {below}.')
if below == 0:
    print('The threshold is therefore not exercised by the real data; the boundary cases (13 versus 14 days) are covered by the tests.')
acceptance['A mean is NULL exactly when its valid-day count is below 14'] = all(row[3] == 0 for row in threshold_rows)

In [ ]:
# Section 7: daily and weekly gold versus a previous (pre-cohort, INC-007) run, when it exists.
previous_db = PREVIOUS_RUN_DIR / 'wearable.duckdb'
if previous_db.is_file():
    previous = duckdb.connect(str(previous_db), read_only=True)
    result_rows = []
    try:
        for table in ('gold_clinical_daily', 'gold_biostat_weekly'):
            before = previous.execute(f'SELECT * FROM {table} ORDER BY ALL').fetchall()
            before_cols = [r[0] for r in previous.execute(
                f"SELECT column_name FROM information_schema.columns WHERE table_name = '{table}' ORDER BY ordinal_position").fetchall()]
            after = connection.execute(f'SELECT * FROM {table} ORDER BY ALL').fetchall()
            after_cols = [r[0] for r in connection.execute(
                f"SELECT column_name FROM information_schema.columns WHERE table_name = '{table}' ORDER BY ordinal_position").fetchall()]
            same = before == after and before_cols == after_cols
            result_rows.append((table, len(before), len(after), same, status(same)))
    finally:
        previous.close()
    render_table('Existing gold tables versus the previous run', ['table', 'rows before', 'rows after', 'identical', 'status'], result_rows)
    acceptance['gold_clinical_daily and gold_biostat_weekly are identical to the previous run'] = all(row[3] for row in result_rows)
else:
    print('Previous run not found at', previous_db, ': comparison NOT RUN.')
    acceptance['gold_clinical_daily and gold_biostat_weekly are identical to the previous run'] = None

In [ ]:
# Section 8: rerun on an isolated copy of the run, then check raw-data immutability.
# The comparison uses the compare command's library function: every table (including the cohort table) and the report bytes.
connection.close()
WORK = Path(tempfile.mkdtemp(prefix='inc008-review-'))
rerun_dir = WORK / 'rerun'
shutil.copytree(RUN_DIR, rerun_dir, ignore=shutil.ignore_patterns('*.wal'))
rerun_result = run_pipeline(DATA_DIR, rerun_dir)
rerun_compare = compare_outputs(RUN_DIR, rerun_dir)
render_table(
    'Same-input rerun',
    ['check', 'result', 'status'],
    [
        ('Every table and the report identical to the supplied run', format_comparison(rerun_compare).strip(), status(rerun_compare.identical)),
        ('Run ID unchanged', rerun_result.run_id, status(rerun_result.run_id == run_id)),
    ],
)
acceptance['Rerun preserves every table (including gold_chronotype_cohort), the report and the run id'] = (
    rerun_compare.identical and rerun_result.run_id == run_id)
shutil.rmtree(rerun_dir, ignore_errors=True)

check_db = duckdb.connect(str(DATABASE_PATH), read_only=True)
manifest = {
    row[0]: row[1]
    for row in check_db.execute('SELECT relative_path, sha256 FROM raw_manifest').fetchall()
}
check_db.close()
hash_mismatches = [
    path for path, digest in manifest.items()
    if hashlib.sha256((DATA_DIR / path).read_bytes()).hexdigest() != digest
]
print('Raw files differing from the run manifest SHA-256:', hash_mismatches[:5] or 'none')
acceptance['Raw files match the run manifest hashes'] = not hash_mismatches

In [ ]:
# Section 9: one corrected record. Copy the data and the supplied run, correct one P001 steps record (+37 steps at
# 2026-01-20T12:00:00), run incremental mode on the copy and a full rebuild into a second directory.
data_copy = WORK / 'data'
shutil.copytree(DATA_DIR, data_copy)
steps_path = data_copy / 'wearable_events' / 'P001' / 'steps.json'
records = json.loads(steps_path.read_text())
target = next(i for i, record in enumerate(records) if record['timestamp'] == '2026-01-20T12:00:00')
records[target]['steps'] += 37
steps_path.write_text(json.dumps(records))

incremental_dir = WORK / 'incremental'
shutil.copytree(RUN_DIR, incremental_dir, ignore=shutil.ignore_patterns('*.wal'))
incremental_result = run_incremental(data_copy, incremental_dir)
full_dir = WORK / 'full'
full_result = run_pipeline(data_copy, full_dir)
equal_to_full = compare_outputs(incremental_dir, full_dir)

compare_db = duckdb.connect(':memory:')
compare_db.execute("ATTACH '" + str(RUN_DIR / 'wearable.duckdb') + "' AS before_db (READ_ONLY)")
compare_db.execute("ATTACH '" + str(incremental_dir / 'wearable.duckdb') + "' AS after_db (READ_ONLY)")
try:
    changed_rows = compare_db.execute(
        '''SELECT DISTINCT participant_id FROM (
               (SELECT * FROM before_db.gold_chronotype_cohort EXCEPT ALL SELECT * FROM after_db.gold_chronotype_cohort)
               UNION ALL
               (SELECT * FROM after_db.gold_chronotype_cohort EXCEPT ALL SELECT * FROM before_db.gold_chronotype_cohort)
           ) ORDER BY 1''').fetchall()
    changed_columns = [
        name for name, _ in COHORT_COLUMNS[1:]
        if compare_db.execute(
            'SELECT COUNT(*) FROM before_db.gold_chronotype_cohort b JOIN after_db.gold_chronotype_cohort a '
            'USING (participant_id) WHERE b.' + name + ' IS DISTINCT FROM a.' + name).fetchone()[0]
    ]
finally:
    compare_db.close()
render_table(
    'Corrected record: incremental versus full rebuild',
    ['check', 'observed', 'status'],
    [
        ('Incremental run parsed only the changed file', incremental_result.changed_paths, status(incremental_result.changed_paths == ('wearable_events/P001/steps.json',))),
        ('Incremental output identical to the full rebuild (all tables, report)', format_comparison(equal_to_full).strip(), status(equal_to_full.identical)),
        ('Run id equals the full rebuild run id', incremental_result.run_id, status(incremental_result.run_id == full_result.run_id)),
        ('Exactly one cohort row differs from the supplied run', [row[0] for row in changed_rows], status(changed_rows == [('P001',)])),
        ('Only the steps mean changed', changed_columns, status(changed_columns == ['steps_daily_mean'])),
    ],
)
acceptance['Corrected record: incremental equals a full rebuild'] = equal_to_full.identical and incremental_result.run_id == full_result.run_id
acceptance['Corrected record: exactly one cohort row changes (P001), in the steps mean only'] = (
    changed_rows == [('P001',)] and changed_columns == ['steps_daily_mean'])
shutil.rmtree(WORK, ignore_errors=True)

In [ ]:
# Section 10: acceptance summary.
raw_hash_after = tree_hash(DATA_DIR)
print(f'Raw data fingerprint after: {raw_hash_after[:16]}')
acceptance['Raw data directory unchanged'] = raw_hash_after == RAW_HASH_BEFORE
rows = [
    (name, 'PASS' if result is True else 'FAIL' if result is False else 'NOT RUN')
    for name, result in acceptance.items()
]
rows.append(('Threshold boundary (13 versus 14 days), NULL and conflicting chronotype, participants with no data, window and old-schema cases', 'NOT RUN here; covered by the pipeline tests'))
render_table('INC-008 review summary', ['acceptance check', 'status'], rows)
print('Review is observational; only temporary copies were rerun and they have been removed.')